# Chapter 5 — Franchise Analysis
Use the familiar 50-company data to examine high and persistent cash returns, growing gross margins, and stable gross margins. A strong result is a starting point for investigating competitive advantage, not proof of a durable franchise.

**75-minute class:** load and inspect (10 min), prepare four years (15 min), CFOA (10 min), margin growth (10 min), stability (10 min), static ratios (10 min), compare and discuss (10 min).

The working code is supplied. Run each cell in order, or choose **Runtime → Run all** in Google Colab. Short Kai prompts help you understand or modify one step. Values remain missing when the inputs do not support a calculation. There is no combined franchise score.


## 1. Load the same public Yahoo Finance dataset
Keep `full_history_df`: Chapter 5 needs historical annual observations. Do not reduce the data to the latest year before preparing the four-year window.


In [ ]:
import numpy as np
import pandas as pd

DATA_URL = 'https://raw.githubusercontent.com/x2003xuy/acct3343-public/main/data/acct3343_financial_data_50.csv'
VALIDATION_SNAPSHOT_SHA256 = 'b48ba2400340a2447e781abf82333e993aaf45b55ea15ad9138a0e2b196e1813'
df = pd.read_csv(DATA_URL)
full_history_df = df.copy(deep=True)
print("Rows:", len(df), "Companies:", df["ticker"].nunique())
display(df[["ticker", "company_name", "fiscal_year", "fiscal_period_end"]].head(10))


## 2. Prepare and inspect four annual observations
For each firm, select its four latest saved annual dates and order them from oldest to newest. Require consecutive fiscal-year labels and date gaps of 350–380 days, allowing normal 52/53-week calendars. A gap, short history, invalid date, or duplicate date produces an explicit reason. We never move to older years just to find nonmissing accounting inputs.

The helper functions below perform these checks. All firms remain in `master_df`. Missing or infinite accounting inputs become `NaN`, never zero.

**Kai prompt:** “Explain why four annual observations give three growth periods.”


In [ ]:
NUMERIC_FIELDS = ['fiscal_year', 'total_revenue', 'cost_of_revenue', 'free_cash_flow', 'total_assets', 'ebit', 'total_debt', 'stockholders_equity', 'cash_and_cash_equivalents', 'tax_rate_for_calcs']

def prepare_history(df):
    """Preserve all rows; invalid numeric inputs stay missing."""
    history = df.copy(deep=True)
    if history['ticker'].isna().any():
        raise ValueError('A row lacks a ticker; investigate before grouping companies.')
    history['fiscal_period_end'] = pd.to_datetime(history['fiscal_period_end'], errors='coerce')
    for column in NUMERIC_FIELDS:
        history[column] = pd.to_numeric(history[column], errors='coerce').replace([np.inf, -np.inf], np.nan)
    history['Gross_Profit'] = history['total_revenue'] - history['cost_of_revenue']
    history['Gross_Margin'] = (
        history['Gross_Profit'] / history['total_revenue'].where(history['total_revenue'].gt(0))
    ).replace([np.inf, -np.inf], np.nan)
    return history.sort_values(['ticker', 'fiscal_period_end']).reset_index(drop=True)

def select_window(firm):
    """Use the latest four dates, without moving backward to improve coverage."""
    ordered = firm.sort_values('fiscal_period_end')
    window = ordered.tail(4).copy()
    reasons = []
    # Invalid dates or duplicate records make selection ambiguous; do not discard them.
    if firm['fiscal_period_end'].isna().any():
        reasons.append('Invalid fiscal date in company history')
    if firm['fiscal_period_end'].duplicated().any():
        reasons.append('Duplicate fiscal date in company history')
    if len(window) < 4:
        reasons.append('Fewer than four annual observations')
    years = window['fiscal_year']
    if years.isna().any() or not years.mod(1).eq(0).all():
        reasons.append('Missing or invalid fiscal year')
    elif len(window) > 1 and not years.diff().iloc[1:].eq(1).all():
        reasons.append('Fiscal years are not consecutive')
    gaps = window['fiscal_period_end'].diff().dt.days.iloc[1:]
    if len(gaps) and not gaps.between(350, 380).all():
        reasons.append('Fiscal date gaps are not annual (350-380 days)')
    return window, '; '.join(reasons)


In [ ]:
history_df = prepare_history(full_history_df)
windows = []
records = []
for ticker, firm in history_df.groupby("ticker", sort=True):
    window, reason = select_window(firm)
    windows.append(window)
    latest = window.iloc[-1]
    records.append({
        "ticker": ticker, "company_name": latest["company_name"], "sector": latest["sector"],
        "History_Observations": len(firm), "Window_Observations": len(window),
        "Growth_Periods": len(window) - 1,
        "First_Fiscal_Date": window["fiscal_period_end"].iloc[0],
        "Last_Fiscal_Date": latest["fiscal_period_end"],
        "History_Status": reason or "Eligible",
        "Latest_Date_Valid": not (firm["fiscal_period_end"].isna().any()
                                 or firm["fiscal_period_end"].duplicated().any()),
        **{name: latest[name] for name in NUMERIC_FIELDS if name != "fiscal_year"},
    })
window_df = pd.concat(windows, ignore_index=True)
master_df = pd.DataFrame(records).set_index("ticker", drop=False)
historical = window_df.groupby("ticker", sort=True)
history_ok = master_df["History_Status"].eq("Eligible")
display(master_df[["ticker", "First_Fiscal_Date", "Last_Fiscal_Date",
                   "Window_Observations", "Growth_Periods", "History_Status"]])
display(window_df.loc[window_df["ticker"].eq("AAPL"),
    ["ticker", "fiscal_year", "fiscal_period_end", "total_revenue", "cost_of_revenue", "free_cash_flow"]])


## 3. CFOA: high and persistent cash returns
**CFOA = sum of four annual Free Cash Flow values / final-year Total Assets.**

Use `free_cash_flow` directly and `total_assets` from the latest selected year. The numerator is a four-year total, not an average. Negative FCF is a valid observation. The asset denominator must be positive.

A higher CFOA indicates more accumulated free cash flow relative to the final asset base. The total alone does not prove that returns were high every year: inspect the annual values too. FCF can vary with investment cycles.

**Kai prompt:** “Show me how `sum(min_count=4)` prevents a missing FCF year from being ignored.”


In [ ]:
master_df["FCF_Observations"] = historical["free_cash_flow"].count()
master_df["Sum_FCF"] = historical["free_cash_flow"].sum(min_count=4)
cfoa_ok = history_ok & master_df["FCF_Observations"].eq(4) & master_df["total_assets"].gt(0)
master_df["CFOA"] = (master_df["Sum_FCF"] / master_df["total_assets"].where(cfoa_ok)).where(cfoa_ok)
master_df["CFOA_Reason"] = np.select(
    [~history_ok, master_df["FCF_Observations"].ne(4), ~master_df["total_assets"].gt(0)],
    [master_df["History_Status"], "Missing or nonfinite free cash flow in four-year window",
     "Missing or nonpositive final total assets"], default="Eligible")
display(master_df[["ticker", "FCF_Observations", "Sum_FCF", "total_assets", "CFOA", "CFOA_Reason"]].head(10))


## 4. Gross margins and return growth
**Annual Gross Margin = (Revenue − COGS) / Revenue.** We map revenue to `total_revenue` and COGS to `cost_of_revenue`. The original collection process can use Yahoo Operating Revenue or Reconciled Cost Of Revenue when its preferred field is absent; the saved CSV does not identify those fallbacks row by row.

**Return Growth = (final Gross Margin / initial Gross Margin)^(1/T) − 1**, where **T = 3** for four consecutive annual observations. This is annualized growth in the margin ratio, not sales growth or a change in percentage points. The slide formula image shows a growth factor; subtracting 1 gives the growth rate used in its worked examples.

For a meaningful geometric growth calculation, this notebook requires positive margins in all four years. Zero or negative margins remain visible but receive no geometric growth rate. Missing inputs or nonpositive revenue invalidate that year's margin.

**Kai prompt:** “Explain the difference between growth in gross margin and growth in gross profit dollars.”


In [ ]:
master_df["Margin_Observations"] = historical["Gross_Margin"].count()
# nth/iloc preserves an endpoint even when it is missing; first()/last() can skip NaN.
master_df["Initial_Gross_Margin"] = historical["Gross_Margin"].agg(lambda x: x.iloc[0])
master_df["Final_Gross_Margin"] = historical["Gross_Margin"].agg(lambda x: x.iloc[-1])
master_df["Positive_Margin_Observations"] = historical["Gross_Margin"].agg(lambda x: int(x.gt(0).sum()))
margin_ok = history_ok & master_df["Margin_Observations"].eq(4)
growth_ok = margin_ok & master_df["Positive_Margin_Observations"].eq(4)
margin_ratio = (master_df["Final_Gross_Margin"] / master_df["Initial_Gross_Margin"].where(growth_ok)).where(growth_ok)
master_df["Return_Growth"] = margin_ratio.pow(1 / master_df["Growth_Periods"].where(growth_ok)) - 1
master_df["Return_Growth_Reason"] = np.select(
    [~history_ok, master_df["Margin_Observations"].ne(4), ~master_df["Positive_Margin_Observations"].eq(4)],
    [master_df["History_Status"], "Missing revenue/COGS, nonpositive revenue, or nonfinite margin",
     "Geometric growth requires positive gross margins in all four years"], default="Eligible")
display(master_df[["ticker", "Initial_Gross_Margin", "Final_Gross_Margin", "Growth_Periods", "Return_Growth"]].head(10))


## 5. Return stability
**Return Stability = arithmetic mean of four Gross Margins / sample standard deviation of those Gross Margins.** Use `ddof=1`.

For firms with positive mean margins, higher values mean less dispersion relative to the mean. A constant margin has zero standard deviation, so this ratio is undefined and is left missing. A very small standard deviation can produce a very large result. A negative mean gives a negative ratio and needs separate interpretation; a stable loss-making margin is not evidence of a strong franchise.

**Kai prompt:** “Why does the sample standard deviation use `ddof=1`?”


In [ ]:
master_df["Mean_Gross_Margin"] = historical["Gross_Margin"].mean().where(margin_ok)
master_df["Sample_SD_Gross_Margin"] = historical["Gross_Margin"].std(ddof=1).where(margin_ok)
nonconstant = historical["Gross_Margin"].nunique().gt(1)
stability_ok = margin_ok & master_df["Sample_SD_Gross_Margin"].gt(0) & nonconstant
master_df["Return_Stability"] = (master_df["Mean_Gross_Margin"] /
    master_df["Sample_SD_Gross_Margin"].where(stability_ok)).where(stability_ok)
master_df["Return_Stability_Reason"] = np.select(
    [~history_ok, master_df["Margin_Observations"].ne(4), ~stability_ok],
    [master_df["History_Status"], "Missing revenue/COGS, nonpositive revenue, or nonfinite margin",
     "Zero sample standard deviation; ratio undefined"], default="Eligible")
display(master_df[["ticker", "Mean_Gross_Margin", "Sample_SD_Gross_Margin", "Return_Stability"]].head(10))


## 6. Separate latest-year measures: gross profitability and Return on Capital
These static measures use the actual latest year. They can remain available even when a firm lacks four years of history.

| Measure | Formula and Yahoo fields |
|---|---|
| Gross Profitability | (`total_revenue` − `cost_of_revenue`) / `total_assets` |
| Capital | `total_debt` + `stockholders_equity` − `cash_and_cash_equivalents` |
| ROC_PreTax | `ebit` / Capital |
| ROC_AfterTax | `ebit` × (1 − `tax_rate_for_calcs`) / Capital |

**Definition notes:** Slide 6 labels its asset-scaled measure “Gross Profit Margin.” We label it **Gross Profitability** to distinguish it from the revenue-scaled gross margins used above. Parentheses apply to Revenue − COGS before division.

Slide 6 shows both EBIT(1−tax)/(debt + equity − cash) and EBIT/Capital. We report both as separate versions using the same explicit capital base; they are not algebraically equal. Yahoo Total Debt is the available book-debt proxy, and Cash And Cash Equivalents excludes other short-term investments. `tax_rate_for_calcs` is Yahoo's calculation-rate proxy, not a verified marginal tax rate. No alternative tax rate or debt definition is silently substituted. We require positive capital and a tax-rate proxy between 0 and 1.

Higher positive ratios suggest more profit relative to assets or capital, but accounting policies, sectors, negative equity, and a small capital base affect comparisons. Inspect the denominator.

**Kai prompt:** “Explain why Gross Profitability and Gross Margin can tell different stories.”


In [ ]:
latest_ok = master_df["Latest_Date_Valid"]
gross_profit = master_df["total_revenue"] - master_df["cost_of_revenue"]
gp_ok = latest_ok & gross_profit.notna() & master_df["total_assets"].gt(0)
master_df["Gross_Profitability"] = (gross_profit / master_df["total_assets"].where(gp_ok)).where(gp_ok)
master_df["Gross_Profitability_Reason"] = np.select(
    [~latest_ok, ~gp_ok], ["Ambiguous latest fiscal date", "Missing revenue/COGS or missing/nonpositive total assets"], default="Eligible")

master_df["Capital"] = master_df["total_debt"] + master_df["stockholders_equity"] - master_df["cash_and_cash_equivalents"]
roc_inputs = master_df[["ebit", "total_debt", "stockholders_equity", "cash_and_cash_equivalents"]].notna().all(axis=1)
roc_ok = latest_ok & roc_inputs & master_df["Capital"].gt(0)
master_df["ROC_PreTax"] = (master_df["ebit"] / master_df["Capital"].where(roc_ok)).where(roc_ok)
master_df["ROC_PreTax_Reason"] = np.select(
    [~latest_ok, ~roc_inputs, ~master_df["Capital"].gt(0)],
    ["Ambiguous latest fiscal date", "Missing EBIT, debt, equity, or cash", "Nonpositive capital (debt + equity - cash)"], default="Eligible")
tax_ok = master_df["tax_rate_for_calcs"].between(0, 1)
master_df["ROC_AfterTax"] = (master_df["ebit"] * (1 - master_df["tax_rate_for_calcs"]) /
    master_df["Capital"].where(roc_ok & tax_ok)).where(roc_ok & tax_ok)
master_df["ROC_AfterTax_Reason"] = np.select(
    [~roc_ok, ~tax_ok], [master_df["ROC_PreTax_Reason"], "Missing tax-rate proxy or rate outside [0, 1]"], default="Eligible")


## 7. Coverage and final company table
All firms stay visible. `NaN` means unavailable, not zero and not the worst firm. Fiscal dates differ across companies. Saved Yahoo statements can be restated and do not supply historical filing-availability dates; these results are not a point-in-time backtest.

Inspect `window_df` for annual inputs and `master_df` for denominators and reasons. The count table below reports the data actually loaded from GitHub, so it remains useful if the source changes.


In [ ]:
METRICS = ['CFOA', 'Return_Growth', 'Return_Stability', 'Gross_Profitability', 'ROC_PreTax', 'ROC_AfterTax']
for metric in METRICS:
    bad_result = ~np.isfinite(master_df[metric]) & master_df[metric + "_Reason"].eq("Eligible")
    master_df.loc[bad_result, metric + "_Reason"] = "Nonfinite calculated result"
    master_df[metric] = master_df[metric].replace([np.inf, -np.inf], np.nan)

coverage = pd.DataFrame({"Valid_Firms": master_df[METRICS].count(),
                         "Missing_Firms": master_df[METRICS].isna().sum()})
display(coverage)
reason_columns = [metric + "_Reason" for metric in METRICS]
display(master_df.loc[master_df[METRICS].isna().any(axis=1), ["ticker"] + reason_columns])
final_columns = ["company_name", "ticker"] + METRICS + [
    "First_Fiscal_Date", "Last_Fiscal_Date", "Window_Observations", "Growth_Periods",
    "FCF_Observations", "Margin_Observations", "History_Status"] + reason_columns
final_results = master_df[final_columns].reset_index(drop=True)
display_columns = ["company_name", "ticker"] + METRICS + [
    "First_Fiscal_Date", "Last_Fiscal_Date", "Window_Observations", "Growth_Periods"]
display(final_results[display_columns].round({metric: 4 for metric in METRICS}))


### Sort and interpret
Change `sort_by` to any metric listed in the coverage table. The display rounds results but the underlying calculations retain full precision. Return growth of `0.02` means about 2% per growth period. CFOA and static ratios are decimal ratios; stability is a multiple.

**Kai prompt:** “Help me sort by Return_Stability and show firms with missing values last.”

Choose two firms and inspect their annual margins and cash flow. Does each measure support the same conclusion? What sector or accounting differences need investigation? Do not add these ratios into an unsupported composite ranking.


In [ ]:
sort_by = "CFOA"  # Try "Return_Growth", "Return_Stability", or "Gross_Profitability"
display(final_results[display_columns].sort_values(sort_by, ascending=False, na_position="last").round({metric: 4 for metric in METRICS}))


## 8. Verify and export
The following checks confirm the table keeps every company, eligible histories contain four years and three growth periods, finite results have an eligibility label, and missing results have a reason. The private reference materials also test the lecture's numerical examples and synthetic edge cases.


In [ ]:
assert len(final_results) == full_history_df["ticker"].nunique()
assert final_results["ticker"].is_unique
assert master_df.loc[history_ok, "Window_Observations"].eq(4).all()
assert master_df.loc[history_ok, "Growth_Periods"].eq(3).all()
for metric in METRICS:
    assert np.isfinite(master_df[metric].dropna()).all()
    assert master_df[metric].notna().equals(master_df[metric + "_Reason"].eq("Eligible"))
final_results.to_csv("ACCT3343_Chapter5_final_results.csv", index=False)
print("Chapter 5 checks passed. Exported all", len(final_results), "companies.")
